In [1]:
import torch

if torch.cuda.is_available():
    print("MANTAP! GPU tersedia dan siap digunakan.")
    print("Nama GPU Anda:", torch.cuda.get_device_name(0))
else:
    print("WADUH! GPU tidak terdeteksi. Colab Anda saat ini masih memakai CPU biasa.")


MANTAP! GPU tersedia dan siap digunakan.
Nama GPU Anda: Tesla T4


In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from google.colab import drive
drive.mount('/content/drive')


MODEL_NAME = "indobenchmark/indobert-base-p1"
CSV_PATH = "/content/drive/MyDrive/PA 3 KEL 10-20260406T061058Z-3-001/PA 3 KEL 10/Dataset/train_dataset_baru.csv"
OUTPUT_DIR = "/content/drive/MyDrive/PA 3 KEL 10-20260406T061058Z-3-001/PA 3 KEL 10/Dataset/pa3_indobert_final_v2"


print("1. Membaca Dataset...")
df = pd.read_csv(CSV_PATH)

df = df[df['final_level'] != 3]

texts = df['text_clean'].astype(str).tolist()
labels = df['final_level'].astype(int).tolist()

print(f"Total data (Label 0, 1, 2): {len(texts)}")

print("2. Membagi Data Training & Validasi")
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Load Tokenizer & Model IndoBERT
print("3. Memuat Model IndoBERT")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

#tokenisasi Dataset
print("4. Tokenisasi Teks")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class IndoBERTDateset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IndoBERTDateset(train_encodings, train_labels)
val_dataset = IndoBERTDateset(val_encodings, val_labels)

print("5. Menyiapkan Konfigurasi Pelatihan")
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,             
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",       
    save_strategy="epoch",
    load_best_model_at_end=True     
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("6. Memulai Proses Training")
trainer.train()

print(f"7. Menyimpan model final ke folder: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Selesai! Silakan download folder pa3_indobert_final_v2 ke laptop Anda.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Membaca Dataset...
Total data (Label 0, 1, 2): 4766
2. Membagi Data Training & Validasi
3. Memuat Model IndoBERT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4. Tokenisasi Teks
5. Menyiapkan Konfigurasi Pelatihan


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

7. Menyimpan model final ke folder: /content/drive/MyDrive/PA 3 KEL 10-20260406T061058Z-3-001/PA 3 KEL 10/Dataset/pa3_indobert_final_v2


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai! Silakan download folder pa3_indobert_final_v2 ke laptop Anda.
